In [2]:

import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import networkx as nx
import random
import heapq
import collections

### All the simulation uses hours and kilometers

In [3]:
LAMBDA_T = [314.2, 162.4, 138.6, 148.8, 273.2, 1118.8, 2773.8, 4036.2, 4237.4, 3277.0, 2843.0, 2876.4, 3143.0, 3277.8, 3546.2, 4335.0, 4945.4, 4525.8, 2847.8, 1828.0, 1378.4, 1271.2, 1171.2, 767.6 ]

In [19]:
#Sampling arrival times of cars to network
def lambdat(t : np.array):
    lambdat = []
    for time in t:
        lambdat.append(LAMBDA_T[int(np.floor(time))])
    return lambdat

def arrival_times(lam): #Taken from lecture notes
    max_T = 24
    arrival_times = collections.deque()
    exp_dist = stats.expon(scale = 1/lam)
    t = exp_dist.rvs()
    while t < max_T:
        arrival_times.append(t)
        t += exp_dist.rvs()
    
    return np.asarray(arrival_times)

In [20]:
Graph = nx.read_gml('./data/networkAssignment.gml')
JUNCTIONS = list(Graph.nodes)

In [21]:
for e in Graph.edges:
    Graph.edges[e]['accident'] = False
    Graph.edges[e]['accident_duration'] = 0

In [36]:
## ADDED THIS FOR DEBUGGING PURPOSE ONLY ##

selected_edges = random.sample(list(Graph.edges), 3)

for u, v in selected_edges:
    # Set the accident attributes for each selected edge
    Graph[u][v]['accident'] = True
    Graph[u][v]['accident_duration'] = 4



In [37]:
Graph.edges[('1410566272', '8432860337')]

{'name': 'Knooppunt Gouwe->Knooppunt Terbregseplein',
 'highway': 'motorway_link',
 'length': 11557.0,
 'lanes': 2,
 'accident': False,
 'accident_duration': 0}

In [38]:
class FES:
    def __init__(self):
        self.events = []

    def add(self, event):
        heapq.heappush(self.events, event)
    
    def next(self):
        return heapq.heappop(self.events)
    
    def isEmpty(self):
        return len(self.events) == 0
    
    def __repr__(self):
        string = ''
        sorted_events = sorted(self.events)
        for event in sorted_events:
            string += f'{event}\n'
        return string

In [39]:
class Event:
    TYPE = ['New car', 'Car departure', 'Accident']
    def __init__(self, typ:int, time, car = None, road = None):
        #types:
            #0 : Arrival of car to the network
            #1 : Car leaves current road and goes on to the next
            #2 : Accident in road
        self.type = typ
        self.time = time
        self.road = road

        if typ == 0:
            car = Car(time_entrance = time)
    
        self.car = car
        
    def __str__(self):
        if self.type == 0:
            return f'{self.TYPE[self.type]} from {self.car.origin} to {self.car.destination} at {self.time}'
        if self.type == 1:
            return f'{self.TYPE[self.type]} of {self.car} at {self.time}h'
        if self.type == 2:
            return f'{self.TYPE[self.type]} at {self.road} at {self.time}h'

    def __lt__(self, other):
        return self.time < other.time

In [45]:
class Car:
    VELOCITIES = [100, 80]
    VELOCITIES_P = [0.9, 0.1]
    def __init__(self, time_entrance, origin = None, destination = None):
        #Origin and destination
        origin, destination = np.random.choice(JUNCTIONS, 2, replace = False)

        self.origin = origin
        self.destination = destination

        #path to follow
        self.path = nx.shortest_path(Graph, self.origin, self.destination, weight = 'length')

        #Velocity
        self.velocity = np.random.choice(self.VELOCITIES, p=self.VELOCITIES_P)

        #Variable to keep track how far into the path we are (to simplify scheduling events)
        #Int between 0 and len(path) - 1 that indicates in which edge we are, starting at 0
        #Essentially, how many edges has it travelled so far
        self.progress = 0

        #Time entrance
        self.time = time_entrance

        #Give it nav with 10% chance
        self.has_nav = np.random.choice([True, False], p=[0.1,0.9])

        #Schedule next event and store it as attribute
        self.next_event = self.schedule_event_exit()


    def __str__(self):
        return f'Vehicle travelling from {self.origin} to {self.destination} at {self.velocity} km/h, atm at {self.path[self.progress]}'


    def custom_weight(self,u,v,data):
        """
        :param u: std for accepting function as weight, node 1
        :param v:  std for accepting function as weight, node 2
        :param data: std for accepting function as weight, edge
        :return: time_to_travel + delay
        """
        length = data['length']
        time_to_travel = self.calc_time_to_travel(length)
        delay = data['accident_duration']
        return time_to_travel + delay


    def calc_time_to_travel(self,length):
        """
        :param length: edge length
        :return: returns time to traverse length based on normal speed
        """
        mean = length / (self.velocity /3.6) #seconds
        std = mean / 20
        time_to_travel = np.random.normal(loc = mean, scale = std) / 3600 #back to hours
        return time_to_travel

    def schedule_event_exit(self):
        if  self.progress < len(self.path) - 1:
            #find next edge to travel through and its length
            #Breaking it down for readability
            current_node = self.path[self.progress]
            next_node = self.path[self.progress + 1]

            edge = Graph.edges[(current_node,next_node)]

            #checking for accident, if accident not found then false
            if self.has_nav and edge['accident']:
                # print("Found an edge with accident")
                try:
                    # print(f"Prev path = {self.path}")

                    new_path = nx.shortest_path(Graph, current_node, next_node, weight = self.custom_weight)
                    # print(f"new path = {new_path}")
                    self.path = new_path
                    self.progress = 0

                    next_node = new_path[self.progress + 1]
                    edge = Graph.edges[(current_node,next_node)]
                    length = edge['length']
                    #Sample travel time of edge
                    time_to_travel = self.calc_time_to_travel(length) + edge['accident_duration']
                except Exception as e:
                    print('Something went wrong in calculating new route..')
                    #Default to following the path anyway
                    length = edge['length']
                    time_to_travel = self.calc_time_to_travel(length) # Helper function
            else:
                length = edge['length']
                time_to_travel = self.calc_time_to_travel(length)

            new_time = self.time + time_to_travel

            #Store event and increase progress
            self.next_event = Event(1 , new_time, car=self)
            self.increase_progress()
            self.increase_time(new_time)

            return self.next_event

        # if self.progress == len(self.path) - 1:
        #     print('Car has reached its destination')
        #     self.travel_time = self.next_event.time

    def increase_progress(self):
        self.progress += 1

    def increase_time(self, new_time):
        self.time = new_time

In [46]:
#Using a thining approach
max_lambda = np.max(LAMBDA_T) + 1
all_arrivals = arrival_times(max_lambda)

uniform_dist = stats.uniform(0,1)
u_rvs = uniform_dist.rvs(len(all_arrivals))
accept_filter = u_rvs * max_lambda < lambdat(all_arrivals)

accepted_arrivals = all_arrivals[accept_filter]

In [47]:
#Simulation (can be turned into an object later)
LIST_CARS = []
fes = FES()
for arrival in accepted_arrivals:
    #Two events associated with each arrival
    arrival_event = Event(0, arrival)
    fes.add(arrival_event)

In [48]:
t = 0 #current time
while t < 24.0:
    event = fes.next()
    t = event.time

    if event.type == 0:
        car_travel_event = event.car.next_event
        fes.add(car_travel_event)
        LIST_CARS.append(event.car)

    if event.type == 1:
        next_travel_event_car = event.car.schedule_event_exit()
        if type(next_travel_event_car) == Event: #If the car has arrived to its destination it wont return an event object
            fes.add(next_travel_event_car)
        # else:
            # print(event.car)

    # if event.type == 2: NO ACCIDENTS YET

In [49]:
#Checking if cars make it to destionation
for i in range(0, len(LIST_CARS)):
    car_i = LIST_CARS[i]
    # if car_i.progress != len(car_i.path) - 1:
    #     print(f'oh oh {LIST_CARS[i].time}')
    if car_i.path[car_i.progress] != car_i.destination:
        print(f'oh oh {LIST_CARS[i].time}')

oh oh 0.3202903405590523
oh oh 0.8091533238799125
oh oh 0.504438361686352
oh oh 0.7254379056984058
oh oh 1.1017210389593985
oh oh 1.5419539030675744
oh oh 1.6802178100508336
oh oh 1.6523006759469943
oh oh 2.342496599836304
oh oh 2.6403149282862577
oh oh 2.6393912462420834
oh oh 3.5106872238777664
oh oh 3.887542366141963
oh oh 4.178711617834895
oh oh 4.407392210959798
oh oh 4.279157646013009
oh oh 4.51170041681927
oh oh 4.993696782924051
oh oh 4.786575425633311
oh oh 5.201803737176166
oh oh 5.488299550011078
oh oh 5.7000442409262355
oh oh 5.554692123711936
oh oh 5.860800325125282
oh oh 5.533407206204462
oh oh 5.994700940144851
oh oh 6.036940466933213
oh oh 6.004383462002824
oh oh 6.082007346958929
oh oh 6.296530221889919
oh oh 6.233348123014049
oh oh 6.365818597212511
oh oh 6.373978398395556
oh oh 6.399075678797799
oh oh 6.506601129793495
oh oh 6.726131390000177
oh oh 6.268078756297448
oh oh 6.24680024246533
oh oh 6.965347208275678
oh oh 6.805069735479783
oh oh 6.800084369725301
oh oh 6